<a href="https://colab.research.google.com/github/l21141431-glitch/ISLP_labs/blob/main/WebScraping_Practica_darel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# Celda 1: Instalacion de librerias y ChromeDriver compatible
!pip install requests beautifulsoup4 lxml selenium pandas openpyxl

# Agregar repositorio oficial de Google Chrome y su clave GPG
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list

!apt-get update # Actualizar despues de agregar el nuevo repositorio

# Instalar Google Chrome
!apt-get install -y google-chrome-stable

# Verificar la ruta de Google Chrome
!which google-chrome

# Obtener la version completa de Google Chrome
chrome_output = !google-chrome --version
if chrome_output:
    chrome_version_full = chrome_output[0].split(' ')[2]
    chrome_major_version = chrome_version_full.split('.')[0]
    print(f"Detected Google Chrome version: {chrome_version_full}")
else:
    raise Exception("Google Chrome not found or version could not be determined.")

# Buscar la URL de ChromeDriver compatible utilizando la API de Chrome for Testing
chromedriver_url = None
try:
    # Usamos 'known-good-versions-with-downloads.json' para obtener enlaces de descarga directos
    versions_url = "https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json"
    response = requests.get(versions_url)
    response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
    versions_data = response.json()

    # Buscar el ChromeDriver que coincida con la versión de Chrome
    # Iteramos en reversa para encontrar la versión más reciente dentro de la rama principal
    for version_info in reversed(versions_data['versions']):
        if version_info['version'].startswith(f"{chrome_major_version}."):
            for download in version_info['downloads']['chromedriver']:
                if download['platform'] == 'linux64':
                    chromedriver_url = download['url']
                    print(f"Found compatible ChromeDriver URL: {chromedriver_url}")
                    break
        if chromedriver_url:
            break

    if not chromedriver_url:
        raise Exception(f"Could not find compatible ChromeDriver for Chrome version {chrome_version_full}")

    # Descargar y configurar ChromeDriver
    # Usamos -N para no sobrescribir si el archivo ya existe y es el mismo
    !wget -N {chromedriver_url} -P /tmp/

    # El nombre del archivo ZIP descargado podría variar, extraemos el nombre del URL
    downloaded_zip_name = chromedriver_url.split('/')[-1]

    # Descomprimir el archivo ZIP. La carpeta resultante suele ser 'chromedriver-linux64'
    !unzip -o /tmp/{downloaded_zip_name} -d /tmp/

    # Mover el ejecutable 'chromedriver' a una ubicación accesible en el PATH
    # Asumimos que la estructura es '/tmp/chromedriver-linux64/chromedriver'
    !mv /tmp/chromedriver-linux64/chromedriver /usr/local/bin/chromedriver
    !chmod +x /usr/local/bin/chromedriver

    print(f"ChromeDriver compatible with Chrome {chrome_version_full} installed at /usr/local/bin/chromedriver.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching ChromeDriver versions from API: {e}")
    print("Attempting to proceed with a general ChromeDriver version as a fallback.")
    # Fallback si falla la API (ej. sin internet o cambio de URL)
    !wget -N https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/126.0.6478.182/linux64/chromedriver-linux64.zip -P /tmp/
    !unzip -o /tmp/chromedriver-linux64.zip -d /tmp/
    !mv /tmp/chromedriver-linux64/chromedriver /usr/local/bin/chromedriver
    !chmod +x /usr/local/bin/chromedriver
    print(f"Fallback ChromeDriver version 126 installed at /usr/local/bin/chromedriver.")
except Exception as e:
    print(f"An unexpected error occurred during ChromeDriver setup: {e}")

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,208 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 6,950 B in 1s (5,280 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for 

In [3]:
# Verificar instalacion
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
print('Todas las librerias instaladas correctamente')

Todas las librerias instaladas correctamente


## Paso 2


In [ ]:
# Celda 2: Primera solicitud HTTP
import requests
from bs4 import BeautifulSoup

In [ ]:
# Realizar solicitud GET al sitio
url = 'http://books.toscrape.com'
response = requests.get(url)

In [ ]:
# Verificar que la solicitud fue exitosa
print(f'Status Code: {response.status_code}') # 200 = exito
print(f'Encoding: {response.encoding}')
print(f'Tamano del HTML: {len(response.text)} caracteres')

Status Code: 200
Encoding: ISO-8859-1
Tamano del HTML: 51294 caracteres


In [ ]:
# Parsear el HTML con BeautifulSoup
soup = BeautifulSoup(response.text, 'lxml')

In [ ]:
# Ver el titulo de la pagina
print(f'Titulo: {soup.title.string}')

Titulo: 
    All products | Books to Scrape - Sandbox



## Paso 3: extraer Datos de un solo Libro

In [ ]:
# Celda 3: Extraer datos del primer libro
# Encontrar el primer articulo de producto
primer_libro = soup.find('article', class_='product_pod')

In [ ]:
# Extraer titulo (esta en el atributo 'title' del enlace dentro de <h3>)
titulo = primer_libro.h3.a['title']
print(f'Titulo: {titulo}')

Titulo: A Light in the Attic


In [ ]:
# Extraer precio
precio = primer_libro.find('p', class_='price_color').text
print(f'Precio: {precio}')

Precio: Â£51.77


In [ ]:
calificacion = primer_libro.find('p', class_='star-rating')['class'][1]
print(f'Calificacion: {calificacion}')

Calificacion: Three


In [27]:
# Extraer disponibilidad
disponible = primer_libro.find('p', class_='instock availability')
stock = disponible.text.strip() if disponible else 'Sin info'
print(f'Disponibilidad: {stock}')

NameError: name 'primer_libro' is not defined

In [ ]:
# Procesar el precio a formato numerico
precio_numerico = float(precio.replace('Â£', ''))
print(f'Precio numerico: {precio_numerico}')

Precio numerico: 51.77


## Paso 4 extraer todos los libros

In [ ]:
# Celda 4: Extraer todos los libros de la pagina 1
libros = soup.find_all('article', class_='product_pod')
print(f'Libros encontrados en pagina 1: {len(libros)}')


Libros encontrados en pagina 1: 20


In [ ]:
datos = []
for libro in libros:
    titulo = libro.h3.a['title']
    precio_texto = libro.find('p', class_='price_color').text
    # Limpiar precio: quitar simbolo y convertir a float
    precio = float(precio_texto.replace('Â£', '').strip())
    calificacion = libro.find('p', class_='star-rating')['class'][1]
    stock = libro.find('p', class_='instock availability')
    disponible = 'Si' if stock and 'In stock' in stock.text else 'No'
    datos.append({'Titulo': titulo, 'Precio': precio, 'Calificacion': calificacion, 'Disponible': disponible})

In [ ]:
# This cell is redundant and was causing inconsistent keys in the 'datos' list.
# It has been removed as 'todos_los_libros' is now the primary data source.

In [ ]:
# Mostrar primeros 5 resultados de todos los libros
import pandas as pd
df_todos_libros = pd.DataFrame(todos_los_libros)
print(df_todos_libros.head())
print(f'\nTotal registros: {len(df_todos_libros)}')


                                  Titulo  Precio Calificacion Disponible
0                   A Light in the Attic   51.77        Three         Si
1                     Tipping the Velvet   53.74          One         Si
2                             Soumission   50.10          One         Si
3                          Sharp Objects   47.82         Four         Si
4  Sapiens: A Brief History of Humankind   54.23         Five         Si

Total registros: 1000


## Paso 5 scraping con paginacion

In [ ]:
# Celda 5: Scraping de TODAS las paginas (50 paginas)
import time

In [ ]:
todos_los_libros = []
base_url = 'http://books.toscrape.com/catalogue/page-{}.html'


In [ ]:
for pagina in range(1, 51): # 50 paginas
    url = base_url.format(pagina)
    response = requests.get(url)

    if response.status_code != 200:
        print(f'Error al acceder a la página {pagina}: {response.status_code}')
        continue

    soup = BeautifulSoup(response.text, 'lxml')
    libros_pagina = soup.find_all('article', class_='product_pod')

    for libro in libros_pagina:
        titulo = libro.h3.a['title']
        precio_texto = libro.find('p', class_='price_color').text
        # Limpiar precio: quitar simbolo y convertir a float
        precio = float(precio_texto.replace('Â£', '').strip())
        calificacion = libro.find('p', class_='star-rating')['class'][1]
        stock_element = libro.find('p', class_='instock availability')
        disponible = 'Si' if stock_element and 'In stock' in stock_element.text else 'No'
        todos_los_libros.append({'Titulo': titulo, 'Precio': precio, 'Calificacion': calificacion, 'Disponible': disponible})

    print(f'Página {pagina} procesada. Total de libros recolectados: {len(todos_los_libros)}')
    time.sleep(1) # Pequeña pausa para evitar sobrecargar el servidor

Página 1 procesada. Total de libros recolectados: 20
Página 2 procesada. Total de libros recolectados: 40
Página 3 procesada. Total de libros recolectados: 60
Página 4 procesada. Total de libros recolectados: 80
Página 5 procesada. Total de libros recolectados: 100
Página 6 procesada. Total de libros recolectados: 120
Página 7 procesada. Total de libros recolectados: 140
Página 8 procesada. Total de libros recolectados: 160
Página 9 procesada. Total de libros recolectados: 180
Página 10 procesada. Total de libros recolectados: 200
Página 11 procesada. Total de libros recolectados: 220
Página 12 procesada. Total de libros recolectados: 240
Página 13 procesada. Total de libros recolectados: 260
Página 14 procesada. Total de libros recolectados: 280
Página 15 procesada. Total de libros recolectados: 300
Página 16 procesada. Total de libros recolectados: 320
Página 17 procesada. Total de libros recolectados: 340
Página 18 procesada. Total de libros recolectados: 360
Página 19 procesada. To

In [ ]:
# Este bloque fue movido e integrado en la celda del bucle principal (7X30UxVop518)

In [ ]:
soup = BeautifulSoup(response.text, 'lxml')
libros = soup.find_all('article', class_='product_pod')

In [ ]:
for libro in libros:
titulo = libro.h3.a[&#39;title&#39;]
precio_texto = libro.find(&#39;p&#39;, class_=&#39;price_color&#39;).text
precio = float(precio_texto.replace(&#39;\u00a3&#39;, &#39;&#39;).strip())
calificacion = libro.find(&#39;p&#39;, class_=&#39;star-rating&#39;)[&#39;class&#39;][1]
stock = libro.find(&#39;p&#39;, class_=&#39;instock availability&#39;)
disponible = &#39;Si&#39; if stock and &#39;In stock&#39; in stock.text else &#39;No&#39;

IndentationError: expected an indented block after 'for' statement on line 1 (3035408221.py, line 2)

In [ ]:
display(df_todos_libros.describe(include='all'))

,Titulo,Precio,Calificacion,Disponible
count,1000,1000.00000,1000,1000
unique,999,NaN,5,1
top,The Star-Touched Queen,NaN,One,Si
freq,2,NaN,226,1000
mean,NaN,35.07035,NaN,NaN
std,NaN,14.44669,NaN,NaN
min,NaN,10.00000,NaN,NaN
25%,NaN,22.10750,NaN,NaN
50%,NaN,35.98000,NaN,NaN
75%,NaN,47.45750,NaN,NaN


In [ ]:
# This cell is redundant as the `todos_los_libros` list was already fully populated by the multi-page scraping loop.

In [ ]:
import pandas as pd # Ensure pandas is imported
# Pausa de cortesia entre solicitudes (buena practica)
# The 'time.sleep(0.5)' and 'if' block below are likely misplaced here,
# as this cell is not within the main scraping loop (cell 7X30UxVop518)
# where 'pagina' is iterated.
# However, to fix the syntax error as requested:
# time.sleep(0.5) # This sleep is out of context here.
# if pagina % 10 == 0:
#     print(f'Paginas procesadas: {pagina}/50') # Corrected indentation

# Assuming the intent was to display the final DataFrame after all scraping is done:
df_completo = pd.DataFrame(todos_los_libros)
print(f'\nTotal de libros extraidos: {len(df_completo)}')
print(df_completo.head(10))


Total de libros extraidos: 1001
                                              Titulo  Precio Calificacion  \
0                               A Light in the Attic   51.77        Three   
1                                 Tipping the Velvet   53.74          One   
2                                         Soumission   50.10          One   
3                                      Sharp Objects   47.82         Four   
4              Sapiens: A Brief History of Humankind   54.23         Five   
5                                    The Requiem Red   22.65          One   
6  The Dirty Little Secrets of Getting Your Dream...   33.34         Four   
7  The Coming Woman: A Novel Based on the Life of...   17.93        Three   
8  The Boys in the Boat: Nine Americans and Their...   22.60         Four   
9                                    The Black Maria   52.15          One   

  Disponible titulo  precio_gbp calificacion disponible  pagina_origen  
0         Si    NaN         NaN          NaN  

## Paso 6 Guardar datos

In [ ]:
# Celda 6: Guardar en CSV y Excel

In [ ]:
# Guardar en CSV
df_completo.to_csv('libros_scrapeados.csv', index=False, encoding='utf-8-sig')
print('Archivo CSV guardado: libros_scrapeados.csv')

Archivo CSV guardado: libros_scrapeados.csv


In [ ]:
# Guardar en Excel
df_completo.to_excel('libros_scrapeados.xlsx', index=False)
print('Archivo Excel guardado: libros_scrapeados.xlsx')

Archivo Excel guardado: libros_scrapeados.xlsx


In [ ]:
# Estadisticas basicas del dataset
print(f'\n--- Resumen del Dataset ---')
print(f'Registros totales: {len(df_completo)}')
print(f'Columnas: {list(df_completo.columns)}')
print(f'Precio promedio: £{df_completo["precio_gbp"].mean():.2f}')
print(f'Precio maximo: £{df_completo["precio_gbp"].max():.2f}')
print(f'Distribucion de calificaciones:')
print(df_completo['calificacion'].value_counts())


--- Resumen del Dataset ---
Registros totales: 1001
Columnas: ['Titulo', 'Precio', 'Calificacion', 'Disponible', 'titulo', 'precio_gbp', 'calificacion', 'disponible', 'pagina_origen']
Precio promedio: £26.08
Precio maximo: £26.08
Distribucion de calificaciones:
calificacion
Five    1
Name: count, dtype: int64


In [ ]:
# Descargar archivo (en Colab)
from google.colab import files
files.download('libros_scrapeados.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Paso 4.2 Configuracion de Selenium


In [4]:
# Celda 7: Configurar Selenium en Colab (modo headless)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [31]:
# Configurar Chrome en modo headless (sin interfaz grafica)
chrome_options = Options()
chrome_options.add_argument('--headless') # Sin ventana
chrome_options.add_argument('--no-sandbox') # Requerido en Colab
chrome_options.add_argument('--disable-dev-shm-usage') # Estabilidad
chrome_options.add_argument('--disable-gpu') # No necesitamos GPU

# Establecer la ruta del binario de Chrome explícitamente
# Esta es la ubicación estándar para Google Chrome después de apt-get install
chrome_options.binary_location = '/opt/google/chrome/google-chrome'
print(f'Chrome binary location set to: {chrome_options.binary_location}')

Chrome binary location set to: /opt/google/chrome/google-chrome


In [36]:
# Iniciar el navegador

# La ruta de chromedriver ahora es /usr/local/bin/chromedriver despues de la instalacion personalizada
chromedriver_path = '/usr/local/bin/chromedriver'

# Limpiar el cache de Selenium para asegurar que no use binarios de Chrome incompatibles
# Esto es común en entornos como Colab para evitar conflictos de versiones
import shutil
import os # Import the os module
if os.path.exists('/root/.cache/selenium'):
    shutil.rmtree('/root/.cache/selenium')
    print('Selenium cache cleaned.')

service = Service(executable_path=chromedriver_path)
driver = webdriver.Chrome(service=service, options=chrome_options)
print('Navegador Chrome iniciado en modo headless')

Navegador Chrome iniciado en modo headless


In [30]:
print('Buscando el binario real de Google Chrome. Espera un momento...')
!dpkg -L google-chrome-stable | grep -E 'bin/google-chrome$|chrome$' # Busca el ejecutable principal

Buscando el binario real de Google Chrome. Espera un momento...
/opt/google/chrome
/opt/google/chrome/chrome
/opt/google/chrome/cron/google-chrome
/opt/google/chrome/google-chrome
/etc/cron.daily/google-chrome


El comando anterior debería haber devuelto la ruta real del binario de Chrome (por ejemplo, `/opt/google/chrome/google-chrome`).

Por favor, *copia esa ruta* (la que termine en `google-chrome`) y la usaremos para actualizar la configuración de Selenium.

**Si no ves ninguna salida o no encuentras una ruta clara, avísame.**

Una vez que tengas la ruta, la usaremos para modificar la celda `ofmn1U70uwvq`.

In [29]:
# Verificar la version de ChromeDriver
!chromedriver --version

ChromeDriver 146.0.7680.165 (4b989da09e15a7dc0de0785cb5ff232aadae3f0f-refs/branch-heads/7680@{#2932})


In [28]:
# Verificar la ruta de Google Chrome
!which google-chrome

# Obtener la version completa de Google Chrome
!google-chrome --version

/usr/bin/google-chrome
Google Chrome 146.0.7680.164 


In [25]:
!find / -name google-chrome 2>/dev/null

In [22]:
# Verificar si el binario de Google Chrome existe en la ruta esperada
!ls -l /usr/bin/google-chrome

ls: cannot access '/usr/bin/google-chrome': No such file or directory


In [21]:
!which google-chrome

In [12]:
print(f'La ruta actual de chromedriver es: {chromedriver_path}')

La ruta actual de chromedriver es: /usr/local/bin/chromedriver


In [13]:
# Verificar si el archivo existe
!ls -l {chromedriver_path}

-rwxr-xr-x 1 root root 16525816 Jul 15  2024 /usr/local/bin/chromedriver


## Paso 4.2 extraer datos

In [32]:
# Celda 8: Scraping con Selenium - Quotes con JavaScript
import time

In [37]:
url = 'http://quotes.toscrape.com/js/'
driver.get(url)

In [38]:
# Esperar a que cargue el contenido dinamico
time.sleep(3) # Espera simple

In [39]:
# Tambien puedes usar espera explicita (mejor practica):
# WebDriverWait(driver, 10).until(
# EC.presence_of_element_located((By.CLASS_NAME, &#39;quote&#39;))
# )

In [41]:
# Extraer todas las citas
quotes = driver.find_elements(By.CLASS_NAME, 'quote')
print(f'Citas encontradas: {len(quotes)}')

Citas encontradas: 10


In [43]:
datos_citas = []
for quote in quotes:
    texto = quote.find_element(By.CLASS_NAME, 'text').text
    autor = quote.find_element(By.CLASS_NAME, 'author').text
    tags_elems = quote.find_elements(By.CLASS_NAME, 'tag')
    tags = ', '.join([t.text for t in tags_elems])
    datos_citas.append({'Texto': texto, 'Autor': autor, 'Tags': tags})